# Aula 08 — Sistemas Multi-Agente com LangChain (Hands On)

**Deploy de Agentes com LLMs** · Luiz Fernando Braz

Nesta prática construímos a **central de atendimento de uma instituição financeira** como um **sistema multi-agente** com o **LangChain 1.0**. Em vez de um único atendente que sabe tudo, temos três agentes **especialistas**, cada um com sua própria **base de conhecimento**:

- **Cartão de Crédito** — fatura, anuidade, limite, bloqueio;
- **Conta Corrente** — PIX, tarifas, extrato, cheque especial;
- **Help Desk (TI)** — login, senha, token, aplicativo.

Sobre esses especialistas, montamos **duas arquiteturas de orquestração**:

1. **Subagentes** — um agente **coordenador** expõe cada especialista como *ferramenta* e decide, a cada atendimento, quem chamar (podendo acionar **vários** para uma pergunta que cruza temas). Roteamento **agêntico** (feito pelo LLM).
2. **Router (orquestrador)** — seguindo o tutorial *router knowledge base* do LangChain: um **classificador** com *saída estruturada* divide a mensagem em **sub-perguntas por especialista**; o grafo dispara os especialistas em **paralelo** (`Send`); e um passo de **Synthesize** combina tudo numa resposta única. Roteamento **determinístico**.

É o padrão **orquestrador-worker** que estudamos no SAIBA MAIS, agora em código real.

> Referência: LangChain — *Multi-agent* e *Router: multi-source knowledge base* (`docs.langchain.com/oss/python/langchain/multi-agent`). Os exemplos chamam o **Claude** de verdade — defina `ANTHROPIC_API_KEY` no `.env`.
>
> As ferramentas imprimem a **ação** que executam (qual especialista foi acionado, o que consultou na base) — assim você *vê* o fluxo multi-agente acontecendo.

## Ambiente e modelo

Carregamos o `.env` e instanciamos **um** modelo de chat, reutilizado por todos os agentes. É *provider-agnóstico*: padrão **Claude**; use `LLM_PROVIDER=openai` para a OpenAI.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

PROVIDER = os.getenv("LLM_PROVIDER", "anthropic")
MODELO = os.getenv("LLM_MODEL") or ("claude-sonnet-4-6" if PROVIDER == "anthropic" else "gpt-4o-mini")
MODEL_STRING = f"{PROVIDER}:{MODELO}"

from langchain.chat_models import init_chat_model

modelo = init_chat_model(MODEL_STRING, temperature=0)

_key = "ANTHROPIC_API_KEY" if PROVIDER == "anthropic" else "OPENAI_API_KEY"
print(f"Modelo: {MODEL_STRING}")
print("API key:", "OK" if os.getenv(_key) else f"FALTANDO -> defina {_key} no .env para rodar os exemplos.")

import textwrap

def imprimir_resposta(titulo, texto, largura=100):
    """Imprime um texto longo com quebras de linha, preservando paragrafos (leitura no notebook)."""
    print(titulo)
    for paragrafo in texto.split("\n"):
        print(textwrap.fill(paragrafo, width=largura) if paragrafo.strip() else "")

## Os especialistas (e suas bases de conhecimento)

Cada especialista é um agente (`create_agent`) que atende **apenas o seu tema** e conta com uma ferramenta de **base de conhecimento** (as políticas e FAQs do banco). A ferramenta faz o especialista ser, de fato, *agêntico*: ele **consulta** a base antes de responder, em vez de inventar — e imprime **o que consultou**, para você acompanhar.

In [ ]:
from langchain.tools import tool
from langchain.agents import create_agent

# Bases de conhecimento (FAQs/politicas) por dominio. Chave = palavra-chave; valor = resposta oficial.
BASE = {
    "cartao": {
        "anuidade": "O cartao Gold tem anuidade de R$ 480/ano (12x); isenta com gasto mensal acima de R$ 2.000.",
        "fatura":   "A fatura fecha no dia 25 e vence no dia 5. A 2a via esta no app em Cartoes > Faturas.",
        "limite":   "O aumento de limite pode ser pedido no app apos 3 faturas pagas em dia; analise em ate 2 dias uteis.",
        "bloqueio": "Em perda/roubo, bloqueie pelo app (Cartoes > Bloquear) ou pela central 0800; 2a via em 7 dias uteis.",
    },
    "conta": {
        "pix":            "Transferencias via PIX sao gratuitas e instantaneas, 24/7. Cadastre chaves em PIX > Minhas chaves.",
        "tarifa":         "A conta Digital e isenta de tarifa de manutencao; TEDs custam R$ 10 (isentos no pacote Premium).",
        "cheque especial":"O cheque especial tem juros de 8% a.m.; pode ser desativado no app em Conta > Cheque especial.",
        "extrato":        "Saldo e extrato dos ultimos 90 dias ficam em Conta > Extrato; periodos maiores, via Atendimento.",
    },
    "helpdesk": {
        "login":     "Se nao consegue logar, atualize o app e use 'Esqueci a senha'. Apos 3 erros a conta bloqueia por 30 min.",
        "senha":     "A senha do app tem 6 digitos; redefina em Perfil > Seguranca > Alterar senha (validacao por SMS).",
        "token":     "O token e gerado no app (Perfil > Token). Se o relogio do celular estiver errado, o token falha.",
        "aplicativo":"Se o app trava ou nao abre, limpe o cache ou reinstale. Requer iOS 15+ / Android 10+.",
    },
}

PERFIS = {
    "cartao":   "Cartao de Credito (fatura, anuidade, limite, bloqueio)",
    "conta":    "Conta Corrente (PIX, tarifas, extrato, cheque especial)",
    "helpdesk": "Help Desk de TI (login, senha, token, aplicativo)",
}

# Nomes curtos e amigaveis para os prints didaticos.
NOMES = {"cartao": "Cartao de Credito", "conta": "Conta Corrente", "helpdesk": "Help Desk"}

def _criar_base(dominio):
    tabela = BASE[dominio]
    @tool
    def consultar_base(assunto: str) -> str:
        """Consulta a base de conhecimento oficial do banco para o tema do especialista."""
        a = assunto.lower()
        for chave, resposta in tabela.items():
            if chave in a:
                print(f"      [base:{NOMES[dominio]}] consultou '{chave}' -> encontrado")
                return resposta
        print(f"      [base:{NOMES[dominio]}] consultou '{assunto}' -> sem entrada")
        return "sem entrada especifica -> oriente de forma geral e sugira a central 0800."
    return consultar_base

def _criar_especialista(dominio):
    prompt = (
        f"Voce e um atendente de um banco, especialista em {PERFIS[dominio]}. "
        "Responda de forma clara e cordial, SEMPRE consultando a base de conhecimento antes. "
        "Atenda apenas o seu tema; se a pergunta for de outro assunto, diga que nao e a sua especialidade."
    )
    return create_agent(model=modelo, tools=[_criar_base(dominio)], system_prompt=prompt)

especialista_cartao   = _criar_especialista("cartao")
especialista_conta    = _criar_especialista("conta")
especialista_helpdesk = _criar_especialista("helpdesk")

ESPECIALISTAS = {
    "cartao": especialista_cartao,
    "conta": especialista_conta,
    "helpdesk": especialista_helpdesk,
}

def _atender(dominio, pergunta):
    """Executa um especialista (identificado pelo dominio) e devolve so o texto final da resposta."""
    print(f"   >> [{NOMES[dominio]}] recebeu: {pergunta}")
    r = ESPECIALISTAS[dominio].invoke({"messages": [{"role": "user", "content": pergunta}]})
    return r["messages"][-1].content

print("3 especialistas criados: Cartao de Credito, Conta Corrente, Help Desk (TI).")

## Arquitetura 1 — Subagentes (coordenador + especialistas como ferramentas)

O **coordenador** é a "recepção" da central: ele não resolve nada sozinho — **decide** quais especialistas acionar para cada atendimento e **agrega** as respostas. Cada especialista vira uma *ferramenta* (`@tool`) que encapsula `agente.invoke(...)`. Como o roteamento passa pelo LLM coordenador, uma pergunta que **cruza temas** pode disparar **mais de um** especialista — e os `print`s deixam esse encaminhamento visível.

In [ ]:
@tool
def atender_cartao_credito(pergunta: str) -> str:
    """Encaminha uma duvida sobre CARTAO DE CREDITO ao especialista dedicado."""
    print("  -> [Coordenador] encaminhando ao especialista de Cartao de Credito")
    return _atender("cartao", pergunta)

@tool
def atender_conta_corrente(pergunta: str) -> str:
    """Encaminha uma duvida sobre CONTA CORRENTE ao especialista dedicado."""
    print("  -> [Coordenador] encaminhando ao especialista de Conta Corrente")
    return _atender("conta", pergunta)

@tool
def atender_help_desk(pergunta: str) -> str:
    """Encaminha uma duvida TECNICA de TI (app, login, senha, token) ao Help Desk."""
    print("  -> [Coordenador] encaminhando ao Help Desk (TI)")
    return _atender("helpdesk", pergunta)

coordenador = create_agent(
    model=modelo,
    tools=[atender_cartao_credito, atender_conta_corrente, atender_help_desk],
    system_prompt=(
        "Voce coordena a central de atendimento de um banco, com tres especialistas: "
        "Cartao de Credito, Conta Corrente e Help Desk de TI. "
        "Para cada mensagem do cliente, identifique o(s) tema(s) e chame a(s) ferramenta(s) adequada(s) "
        "- uma por especialista necessario - e consolide uma resposta unica e organizada. "
        "Nao responda por conta propria; delegue aos especialistas."
    ),
)

pergunta = ("Fui cobrado de uma anuidade no meu cartao e, para piorar, o aplicativo trava "
            "quando tento abrir a fatura. Podem ajudar?")
print("CLIENTE:", pergunta, "\n")
resposta = coordenador.invoke({"messages": [{"role": "user", "content": pergunta}]})
imprimir_resposta("\nRESPOSTA CONSOLIDADA:", resposta["messages"][-1].content)

## Arquitetura 2 — Router / *knowledge base* (classificar → fan-out → Synthesize)

Reproduzimos, no nosso cenário, o tutorial **Router: multi-source knowledge base** do LangChain. O fluxo é um **grafo** do LangGraph:

**START → classificar → [especialistas em paralelo] → sintetizar → END**

1. **Classificar (saída estruturada)** — um `roteador_llm` com `with_structured_output` devolve uma **lista** de `{especialista, sub-pergunta}`: ele não só escolhe os temas como **divide** a mensagem numa pergunta objetiva para cada especialista.
2. **Fan-out com `Send`** — `rotear` transforma cada classificação num `Send(especialista, {"pergunta": sub-pergunta})`; os especialistas rodam **em paralelo** e acumulam no estado via *reducer* `operator.add`.
3. **Synthesize** — `sintetizar` combina as respostas (ex.: **Conta Corrente** + **Help Desk**) numa **resposta única** ao cliente.

Diferente dos subagentes (o LLM decide a cada passo), aqui a estrutura é fixa e o roteamento é feito por um classificador barato — mais **previsível** e fácil de escalar.

In [ ]:
import operator
from typing import Annotated, Literal, TypedDict
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

# LLM dedicado ao roteador (classificacao + sintese). Handle separado, como no guia
# (poderia ser um modelo menor/mais barato que o dos especialistas).
roteador_llm = init_chat_model(MODEL_STRING, temperature=0)

# --- Esquema da classificacao (saida estruturada) ---
class Classificacao(TypedDict):
    especialista: Literal["cartao", "conta", "helpdesk"]
    pergunta: str        # sub-pergunta objetiva, direcionada a esse especialista

class ResultadoClassificacao(BaseModel):
    classificacoes: list[Classificacao] = Field(
        description="Especialistas a acionar, cada um com a sub-pergunta especifica para ele."
    )

# --- Estado do grafo ---
class EstadoRouter(TypedDict):
    mensagem: str                                  # mensagem original do cliente
    classificacoes: list[Classificacao]
    respostas: Annotated[list, operator.add]       # coletadas em paralelo (reducer)
    resposta_final: str                            # saida do Synthesize

class EntradaEspecialista(TypedDict):
    pergunta: str

def classificar(estado: EstadoRouter) -> dict:
    llm_estruturado = roteador_llm.with_structured_output(ResultadoClassificacao)
    resultado = llm_estruturado.invoke([
        {"role": "system", "content":
         "Voce roteia o atendimento de um banco. Analise a mensagem do cliente e liste os especialistas "
         "necessarios (cartao, conta, helpdesk), cada um com uma sub-pergunta objetiva so do seu tema. "
         "Use mais de um especialista quando a mensagem cruzar assuntos."},
        {"role": "user", "content": estado["mensagem"]},
    ])
    print("  -> [Router] classificacao:", [(c["especialista"], c["pergunta"]) for c in resultado.classificacoes])
    return {"classificacoes": resultado.classificacoes}

def rotear(estado: EstadoRouter) -> list[Send]:
    # fan-out: um Send por especialista, levando sua sub-pergunta
    return [Send(c["especialista"], {"pergunta": c["pergunta"]}) for c in estado["classificacoes"]]

def _criar_no(dominio):
    def executar(entrada: EntradaEspecialista) -> dict:
        resposta = _atender(dominio, entrada["pergunta"])
        return {"respostas": [{"especialista": dominio, "resposta": resposta}]}
    return executar

def sintetizar(estado: EstadoRouter) -> dict:
    if not estado["respostas"]:
        return {"resposta_final": "Nao identifiquei um especialista para o seu caso. Procure a central 0800."}
    print(f"  -> [Synthesize] combinando {len(estado['respostas'])} resposta(s) de especialista(s)")
    formatado = [f"[{NOMES[r['especialista']]}]\n{r['resposta']}" for r in estado["respostas"]]
    sintese = roteador_llm.invoke([
        {"role": "system", "content":
         f'Combine as respostas dos especialistas para a mensagem: "{estado["mensagem"]}". '
         "Entregue UMA unica resposta ao cliente, clara e cordial, sem repeticao, organizada por assunto."},
        {"role": "user", "content": "\n\n".join(formatado)},
    ])
    return {"resposta_final": sintese.content}

central = (
    StateGraph(EstadoRouter)
    .add_node("classificar", classificar)
    .add_node("cartao", _criar_no("cartao"))
    .add_node("conta", _criar_no("conta"))
    .add_node("helpdesk", _criar_no("helpdesk"))
    .add_node("sintetizar", sintetizar)
    .add_edge(START, "classificar")
    .add_conditional_edges("classificar", rotear, ["cartao", "conta", "helpdesk"])
    .add_edge("cartao", "sintetizar")
    .add_edge("conta", "sintetizar")
    .add_edge("helpdesk", "sintetizar")
    .add_edge("sintetizar", END)
    .compile()
)

mensagem = ("Preciso fazer um PIX urgente, mas o aplicativo trava quando vou gerar o token "
            "de aprovacao. Como resolvo?")
print("CLIENTE:", mensagem, "\n")
resultado = central.invoke({"mensagem": mensagem, "respostas": []})
imprimir_resposta("\nRESPOSTA COMBINADA (Synthesize):", resultado["resposta_final"])

## Subagentes × Router: qual usar?

- **Subagentes** — o LLM **coordenador** decide dinamicamente quem chamar; pode acionar **vários** especialistas, encadear e reagir ao resultado. Mais **flexível**, porém gasta mais **tokens** (o coordenador raciocina a cada passo). É a forma mais próxima do **orquestrador-worker** do SAIBA MAIS.
- **Router** — um **classificador** (saída estruturada) divide a mensagem em sub-perguntas, os especialistas rodam **em paralelo** (`Send`) e um passo de **Synthesize** entrega a **resposta combinada**. Mais **previsível** e fácil de escalar (adicione um especialista sem tocar nos demais); o custo do LLM concentra-se na classificação e na síntese.

A lição da aula reaparece: multi-agente **não é grátis**. Escolha a arquitetura pelo equilíbrio **custo × controle × flexibilidade** que o problema exige — e meça (tokens, latência, qualidade) antes de generalizar.